In [1]:
!pip install keybert

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 77.7 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling

In [97]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("shuyangli94/food-com-recipes-and-user-interactions")

print("Path to dataset files:", path)

Path to dataset files: /root/.cache/kagglehub/datasets/shuyangli94/food-com-recipes-and-user-interactions/versions/2


In [98]:
from typing import Dict
from pandas import DataFrame
from typing import List
import pandas as pd
from keybert import KeyBERT
import kagglehub

def search_strict(keywords: List, recipe_db: DataFrame) -> Dict:
  recipes_found = {}
  for recipe in keywords:
    if recipe[0] in recipe_db.name.values:
      recipes_found[recipe[0]] = recipe_db.ingredients[recipe_db.name == recipe[0]]
    else:
      print(f"Could not find recipe for {recipe[0]}")
  return recipes_found

In [99]:
recipes = pd.read_csv("/root/.cache/kagglehub/datasets/shuyangli94/food-com-recipes-and-user-interactions/versions/2/RAW_recipes.csv")
path = kagglehub.dataset_download("shuyangli94/food-com-recipes-and-user-interactions")

In [104]:
!ls /root/.cache/kagglehub/datasets/shuyangli94/food-com-recipes-and-user-interactions/versions/2/RAW_recipes.csv

/root/.cache/kagglehub/datasets/shuyangli94/food-com-recipes-and-user-interactions/versions/2/RAW_recipes.csv


In [105]:
#Entire pipeline:
kw_model = KeyBERT()
prompt = "I want to try and make some brownies this week"
recipes = pd.read_csv(f"{path}/RAW_recipes.csv")

keywords_prompt = kw_model.extract_keywords(prompt, keyphrase_ngram_range=(1,2))

recipe_ingredients: Dict = search_strict(keywords_prompt, recipes)

Could not find recipe for make brownies
Could not find recipe for brownies week
Could not find recipe for try make
Could not find recipe for week


In [106]:
keywords_prompt

[('make brownies', 0.8146),
 ('brownies week', 0.8008),
 ('brownies', 0.7016),
 ('try make', 0.3717),
 ('week', 0.2913)]

In [107]:
recipe_ingredients

{'brownies': 30543    ['butter', 'sugar', 'eggs', 'flour', 'cocoa', ...
 Name: ingredients, dtype: object}

In [108]:
recipe_ingredients['brownies'][30543]

"['butter', 'sugar', 'eggs', 'flour', 'cocoa', 'vanilla']"

In [109]:
print(recipes[recipes.name.values == 'brownies'].description.values)


['this was always a favorite at school bake sales when i was young.']


In [110]:
for step in recipes[recipes.name == 'brownies'].steps.values:
  print(recipes.loc[recipes.name == 'brownies', 'steps'].iloc[0])


['combine sugar with melted margarine', 'cream well', 'add eggs and mix well', 'stir in flour and cocoa', 'add vanilla', 'grease and flour a 9 inch round or square cake pan', 'bake approximately 20 minutes @ 350', "best not to over bake them , they shouldn't be dry"]
